# Chapter 1 — memorisation vs relational knowledge

**Settings:** GPU **T4 ×2** · Internet **ON** · Persistence **Variables and Files**

---

### The claim

> Fine-tuning for KGC installs **entity-name memorisation**, not relational structure.
> Measured on WN11: memorisation **0.393** of the 0.4315 above-chance gain — **91%**.

### ★ Primary dataset: YAGO3-10

Its 37 relations are **strongly typed**, which is what the type/rule question needs:

```
wasBornIn      Person  → Place        isCitizenOf   Person  → Country
playsFor       Person  → Club         graduatedFrom Person  → University
hasCapital     Country → City         isMarriedTo   Person  → Person
```

WN11's relations are lexical (`_type_of`, `_has_instance`) — `Person –bornIn→ Location` does not exist there. On YAGO3-10 an induced type *recovers* the semantic type, because the relations are typed.

⚠️ **YAGO3-10 ships no ±1 test labels** (KG-LLM used it for link prediction). Section 2 generates them, and section 3 validates the result before anything trains.

### Three environment rules, learned the hard way

**1 · One GPU per job** — two visible → `DataParallel` → `mat1 and mat2 must have the same dtype`.
**2 · fp16 + `sdpa`** — fp16 + `eager` returns **NaN**, which looks like `train_loss=0.0`, i.e. a finished run.
**3 · Remove `torchao`** — Kaggle ships 0.10.0; transformers refuses to import below 0.16.

## 0 · Setup

In [ ]:
REPO_URL = "https://github.com/lynda-lagh/contribution-.git"
DEST = "/kaggle/working/repo"

import os, subprocess, sys, socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No network. Settings > Internet > ON, then re-run.")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(f"{DEST}/.git"):
    run(["git", "-C", DEST, "fetch", "--all"])
    run(["git", "-C", DEST, "reset", "--hard", "origin/main"])
    print("updated")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, DEST])
    print("cloned")

os.chdir(DEST); sys.path.insert(0, DEST)
print("HEAD:", run(["git", "log", "-1", "--oneline"]))
print("\n\u2605 Does that hash match your latest push?")

In [ ]:
PIN_TRANSFORMERS = "4.57.6"
import json, subprocess, sys

def stack():
    code = ("import json, peft, transformers; print(json.dumps({"
            "'peft': peft.__version__, 'tf': transformers.__version__}))")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else None

s = stack()
if not (s and s["tf"] == PIN_TRANSFORMERS):
    print(f"installing… (have {s})")
    !pip install -q -r requirements.txt
    !pip install -q peft "transformers=={PIN_TRANSFORMERS}"
!pip uninstall -y -q torchao 2>/dev/null
!pip install -q tqdm

import peft, transformers
print(f"\npeft {peft.__version__} | transformers {transformers.__version__}")

In [ ]:
# 22 tests, ~2 s, no GPU. A DIFFERENT random graph each run — the seed is printed.
# Each test prints the prompt it builds: the clearest documentation of P0–P4.
!python -m chapter1.test_chapter1

## 1b · ★★ THE SECOND INSTRUMENT — free, no GPU

Reads an **existing** `ch1_*.json` and asks a different question from anonymisation:
**names stay intact**, we only split by whether the entity was seen in training.

We trained on 10,000 of WN11's 112,581 triples, so only ~35% of entities were ever seen — an accidental inductive split sitting in the results, never read.

★ Two instruments agreeing from opposite directions is a different class of claim from one asserting.

⚠️ It reports a **null** properly. If there is no familiarity gap, it says so — and that *weakens* the memorisation reading. Better to learn that here than in a viva.


In [ ]:
# needs a previous ch1 result + its built test set. Skips cleanly if absent.
!python -m chapter1.seen_unseen --dataset WN11 || echo "(no prior ch1_WN11.json — run after the first evaluation)"


## 1 · Download YAGO3-10

~1M triples, 123,182 entities. The clone streams git's own progress plus a heartbeat every 15 s, so you can see it moving.

In [ ]:
!python -m scripts.fetch_data --datasets YAGO3-10 WN11

## 2 · Give YAGO3-10 its missing ±1 test labels

It ships positives only. Without negatives every gold answer becomes `"No"` and a model scores 100% by always answering No.

The original file is backed up to `test.original.tsv`. Candidates that already exist anywhere in the graph are **rejected**, so only facts absent from train ∪ valid ∪ test can slip through — that rejection count is your closed-world exposure.

★ **Report this:** these negatives are ours, not the benchmark's. Numbers are not comparable to published YAGO3-10 results; all comparisons must be internal.

In [ ]:
!python -m scripts.make_test_negatives --dataset YAGO3-10 --strategy random --seed 42

## 3 · ★ VALIDATE — gate everything on this

Hard checks, non-zero exit on failure:

| | |
|---|---|
| **FATAL** | test labels absent / one-class · train-test leakage · unknown ids · >1% negatives that are true |
| **WARN** | duplicate triples · self-loops · class imbalance · empty descriptions |

For reference, WN11 itself fails one check: 7 of its 10,544 shipped negatives appear in train. A benchmark defect worth a footnote.

In [ ]:
!python -m chapter1.validate --dataset YAGO3-10 WN11

## 4 · Profile — understand the data before designing on it

Answers, per dataset: what do the ids look like · are there real descriptions · how imbalanced are the relations · which type source works and how concentrated is it · how ambiguous are surface forms · what does N training triples cover.

Ends with a **verdict** on which conditions are viable. Both bugs we hit — 100% `OTHER` types, and unlabelled test sets — are visible here in seconds.

In [ ]:
!python -m chapter1.profile_data --dataset YAGO3-10 WN11 --json results/profile.json

### Reading the profile

* **`top type share`** — 22 types sounds healthy, but if one covers 60% the tag carries little. Entropy (bits) is the honest measure.
* **`coverage by training size`** — this sets the seen/unseen split. On WN11 at 10k triples only 22% of test triples have both entities seen; 25k gives ~51%, which is balanced. Choose deliberately.
* **`ambiguity`** — WN11 is 22.7% (worst: `break` ×32). If YAGO3-10 is near zero, disambiguation features have nothing to work with there.

## 5 · Build the conditions

Each build prints an example prompt and asserts the condition **differs from its baseline** — the guard that caught C rendering identically to B.

In [ ]:
DATASET = "YAGO3-10"

# the plain + anonymised test sets that every `gap` measurement needs
!python -m src.data.build_instructions --dataset {DATASET} --n_triples 10000 --seed 42
!python -m src.data.build_instructions --dataset {DATASET} --n_triples 10000 --seed 42 --anonymise

!python -m chapter1.data --all --dataset {DATASET}

## 6 · Train — C and G first, they carry the claim

Each run ends with a **fit verdict** read from the learning curve (overfit / underfit / train-eval gap / good fit) plus an ASCII curve. `load_best_model_at_end` keeps the lowest-eval-loss checkpoint and early stopping halts when it stops improving.

This matters more than usual: the thesis *claims* the model memorises, so an overfit training setup would make that a statement about our hyper-parameters rather than about KGC.

In [ ]:
import subprocess
def pair(a, b):
    """Two INDEPENDENT jobs, one per T4. Never DataParallel."""
    pa = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {a}", shell=True)
    pb = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 {b}", shell=True)
    return pa.wait(), pb.wait()

R = f"python -m chapter1.run --dataset {DATASET} --train --condition"
pair(f"{R} C", f"{R} G")        # ~40 min — the two that carry the claim


In [ ]:
# baselines + ★ the shuffled-names control (S)
# S keeps every real name and only PERMUTES which entity holds which:
#     real  : dog, a domesticated carnivore _hypernym mammal…
#     S     : metamorphic _hypernym lepiota
#     anon  : entity5 _hypernym entity0
# Same vocabulary, same lengths, same readability — only the name↔entity
# BINDING is destroyed. If S ≈ B, "anonymisation removes all signal" is dead.
pair(f"{R} A", f"{R} B")        # ~40 min


In [ ]:
pair(f"{R} S", f"{R} D")        # ★ S is the cheapest insurance in the project


In [ ]:
pair(f"{R} D", f"{R} E")        # ~2 h 10 min — E is 70k instances

## 7 · Evaluate — both test sets, always

Every model scored on the **real** and the **anonymised** test set. The **gap** is the result; a single accuracy cannot express the claim.

Emits the seen/unseen split and calibration-by-familiarity for free.

In [ ]:
for C in ["A", "B", "C", "D", "E", "G", "S"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.run --dataset {DATASET} --evaluate --condition {C}


### ★ SMI — the third instrument, and the wedge against FLAME

SMI alone says the **opposite** of our thesis. Our first run printed:

```
SMI 0.01052 → 0.04742   (+3.5×)
"representation enriched -> tuning installed KNOWLEDGE"
```

Beside the gap it says something sharper:

> SMI rose 3.5× **while** 91% of the accuracy is surface form → **SMI cannot separate memorisation from relational knowledge.** Representations became more label-informative, and the information is the entity *name*.

That is FLAME's limitation, resolved — and it exists only because both are reported together. `evaluate.py` emits a `joint_reading` and flags the case where the two point different ways.

⚠️ Slow: 600 samples × 2 model loads. Off by default. **A and B carry the comparison.**


In [ ]:
for C in ["A", "B"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.run --dataset {DATASET} --evaluate --condition {C} --smi


## 8 · The detailed report

Accuracy alone hides both failure modes here. This prints, per condition:

* confusion matrix · per-class precision/recall/F1 · **balanced accuracy**
* **degenerate check** — is it just always answering Yes? invisible in accuracy
* **majority-class baseline** — beating chance is not enough on a skewed set
* **per-relation** breakdown — YAGO3-10 has 37 relations; a mean can hide 30 failures
* seen/unseen · ECE · Brier · risk–coverage · McNemar

In [ ]:
!python -m chapter1.analysis --dataset {DATASET}
!python -m chapter1.report --dataset {DATASET}

## 9 · Completion task — is this really KGC?

Triple classification completes nothing. Here the classifier becomes a **ranker**: score every candidate tail by `P(Yes | h, r, t)` and sort. No retraining.

★ **This makes MRR computable**, which the spec had recorded as impossible under generative decoding.

✅ **50-way, filtered** — the field standard. RealKGC §4.1: *"Following standard practice like CATS and ToC, we rank each answer against 50 randomly sampled negative entities."* Say so in every caption.

In [ ]:
for C in ["A", "B", "C", "S"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank \
        --adapter checkpoints/ch1-{DATASET}-{C} --dataset {DATASET} \
        --condition {C} --limit 500


## 10 · Cross-dataset comparison — optional, and only if it answers something

Running the ladder on a second dataset is **not** automatically worth 6 more runs. It is worth it here because the two graphs differ in a way that maps onto a hypothesis:

| | WN11 | YAGO3-10 |
|---|---|---|
| entities | 38,588 | **123,182** |
| memorising means | **lexical** — *"dog is a mammal"* | **world-factual** — *"Obama born in Honolulu"* |
| relations | lexical | **typed** |

So the comparison is a **decomposition**, not a replication: if the gap differs, it tells you how much memorisation is word-meaning versus world-fact. And memorisation pressure should scale with vocabulary — 3.2× more entities.

⚠️ If you are short on time, **one dataset done properly beats two done partially.** WN11 A/B already exist (0.9315 / 0.5385).

In [ ]:
# only if time allows — WN11 A and B are already done
# R2 = "python -m chapter1.run --dataset WN11 --train --condition"
# pair(f"{R2} C", f"{R2} G")

## 11 · Package — download before the session ends

In [ ]:
!zip -qr /kaggle/working/chapter1_results.zip results/ data/*/built/manifest.json
!du -sh /kaggle/working/chapter1_results.zip
!ls results/ | head -30

---
### Checklist

* [ ] tests pass · commit hash matches your push
* [ ] `transformers 4.57.6` · `torchao` absent
* [ ] **validate PASSES on YAGO3-10** before any training
* [ ] profile verdict says the type conditions are viable
* [ ] each build printed `[guard] … prompts carry a type tag ✓`
* [ ] every fit verdict read (no silent overfit)
* [ ] every condition has **both** `acc_real` and `acc_anon`
* [ ] ★★ **`seen_unseen` run** — the free second instrument
* [ ] ★★ **condition S run** — closes the "destroys all signal" objection
* [ ] ★ **`--smi` on A and B** — the joint reading against FLAME
* [ ] 3 seeds on **B vs C** — it carries the central claim
* [ ] ranking captions say **50-way, filtered**
* [ ] YAGO3-10 tables note that **negatives are generated, not shipped**
* [ ] `chapter1_results.zip` downloaded

### The four instruments

| | destroys | costs |
|---|---|---|
| anonymisation (A→B) | *all* surface information | 1 run |
| shuffled names (S) | only the name↔entity **binding** | 1 run |
| seen / unseen | **nothing** — names stay intact | **free** |
| SMI | — reads representations directly | slow, no training |

> They fail in different ways, so agreement is evidence rather than repetition.
> A flat result is still a result — but **one instrument with one live objection is not.**
